# Milestone 01 - Bank Account Fraud Dataset Audit

## Project
Bank Account Fraud Detection - Deep Learning & MLOps

## Purpose

This notebook performs the initial audit of `Base.csv` from the Bank Account Fraud Dataset Suite.

The goal is to understand the dataset before making preprocessing, splitting, or modeling decisions.

## Learning Objectives

By the end of this notebook, I should be able to explain:

1. The exact shape and schema of the dataset.
2. The prevalence of the `fraud_bool` target.
3. Which features are numerical versus categorical.
4. Whether missing values or duplicates are present.
5. Which categorical features have high or unusual cardinality.
6. How the `month` feature is distributed.
7. Whether fraud rates change across months.
8. What potential leakage or data-quality risks require deeper investigation.
9. Whether a chronological train/validation/test split may be appropriate.

## Important Rules

- Raw data is read from `data/raw/Base.csv`.
- Raw data will never be modified in place.
- No model training will occur in this notebook.
- No final train/validation/test split will be selected until the audit is complete.
- No preprocessing will be fit before the split strategy is finalized.
- The final test set must remain sealed during model development.

### 1. Environment and reproducibility check

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import matplotlib

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)

print("\nNotebook working directory:")
print(Path.cwd())

Python: 3.11.9
NumPy: 2.4.6
Pandas: 3.0.6
scikit-learn: 1.9.1
Matplotlib: 3.11.2

Notebook working directory:
c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\notebooks


### 2. Define and verify the raw data path

In [2]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Base.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("File exists:", RAW_DATA_PATH.exists())
print("File size (MB):", round(RAW_DATA_PATH.stat().st_size / (1024**2), 2))

Project root: c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops
Raw data path: c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\data\raw\Base.csv
File exists: True
File size (MB): 203.54


### 4. Load the dataset

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

print("Shape:", df.shape)
display(df.head())

Shape: (1000000, 32)


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


### 5. Audit schema and data types

In [4]:
schema_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "unique_count": df.nunique(dropna=False).values,
})

display(schema_summary)

,column,dtype,non_null_count,missing_count,unique_count
0,fraud_bool,int64,1000000,0,2
1,income,float64,1000000,0,9
2,name_email_similarity,float64,1000000,0,998861
3,prev_address_months_count,int64,1000000,0,374
4,current_address_months_count,int64,1000000,0,423
5,customer_age,int64,1000000,0,9
6,days_since_request,float64,1000000,0,989330
7,intended_balcon_amount,float64,1000000,0,994971
8,payment_type,str,1000000,0,5
9,zip_count_4w,int64,1000000,0,6306


### 6. Audit the target class imbalance

In [5]:
target_counts = df["fraud_bool"].value_counts().sort_index()
target_rates = df["fraud_bool"].value_counts(normalize=True).sort_index()

target_summary = pd.DataFrame({
    "count": target_counts,
    "proportion": target_rates,
    "percent": target_rates * 100,
})

display(target_summary)

fraud_rate = df["fraud_bool"].mean()

print(f"Fraud rate: {fraud_rate:.6f}")
print(f"Fraud percentage: {fraud_rate * 100:.3f}%")
print(f"Non-fraud to fraud ratio: {(1 - fraud_rate) / fraud_rate:.1f}:1")

,count,proportion,percent
fraud_bool,,,
0,988971,0.988971,98.8971
1,11029,0.011029,1.1029


Fraud rate: 0.011029
Fraud percentage: 1.103%
Non-fraud to fraud ratio: 89.7:1


## DATASET AUDIT SUMMARY
With 1,000,000 rows, we have approximately 11,029 fraud cases vs. 988,971 non-fraud cases. A model that simply predicts “not fraud” every time would already achieve about 98.9% accuracy, which is why accuracy cannot be our primary metric. This supports our decision to prioritize PR-AUC, while also tracking recall, precision, F1, false positives, false negatives, and threshold-dependent business cost.

### 7. Check for sentinel-coded missing values

In [6]:
numeric_columns = df.select_dtypes(include=np.number).columns

negative_value_summary = pd.DataFrame({
    "minimum_value": df[numeric_columns].min(),
    "negative_count": (df[numeric_columns] < 0).sum(),
    "minus_one_count": (df[numeric_columns] == -1).sum(),
})

negative_value_summary["negative_percent"] = (
    negative_value_summary["negative_count"] / len(df) * 100
)

negative_value_summary = negative_value_summary[
    negative_value_summary["negative_count"] > 0
].sort_values("negative_count", ascending=False)

display(negative_value_summary)

,minimum_value,negative_count,minus_one_count,negative_percent
intended_balcon_amount,-15.530555,742523,0,74.2523
prev_address_months_count,-1.000000,712920,712920,71.2920
bank_months_count,-1.000000,253635,253635,25.3635
credit_risk_score,-170.000000,14445,488,1.4445
current_address_months_count,-1.000000,4254,4254,0.4254
session_length_in_minutes,-1.000000,2015,2015,0.2015
device_distinct_emails_8w,-1.000000,359,359,0.0359
velocity_6h,-170.603072,44,0,0.0044


## MISSING VALUE SUMMARY
Several columns clearly use -1 heavily enough that we must treat it as a possible sentinel value until we confirm the feature definitions. For example, prev_address_months_count is -1 for 71.292% of rows and bank_months_count for 25.364%. We should not automatically replace those yet.
The other negative-valued features need different treatment. intended_balcon_amount, credit_risk_score, and velocity_6h contain negative values that are not simply -1, so those negatives may be legitimate parts of their feature ranges. We need to inspect their distributions instead of assuming they represent missing data.

### 8. Inspect the flagged numeric features

In [7]:
flagged_columns = [
    "intended_balcon_amount",
    "prev_address_months_count",
    "bank_months_count",
    "credit_risk_score",
    "current_address_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w",
    "velocity_6h",
]

distribution_summary = df[flagged_columns].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

distribution_summary["minus_one_count"] = (
    df[flagged_columns] == -1
).sum()

distribution_summary["minus_one_percent"] = (
    distribution_summary["minus_one_count"] / len(df) * 100
)

display(distribution_summary)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,minus_one_count,minus_one_percent
intended_balcon_amount,1000000.0,8.661499,20.236155,-15.530555,-1.831266,-1.588011,-1.181488,-0.830507,4.984176,50.394003,100.826524,112.956928,0,0.0000
prev_address_months_count,1000000.0,16.718568,44.046230,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,12.000000,98.000000,231.000000,383.000000,712920,71.2920
bank_months_count,1000000.0,10.839303,12.116875,-1.000000,-1.000000,-1.000000,-1.000000,5.000000,25.000000,30.000000,31.000000,32.000000,253635,25.3635
credit_risk_score,1000000.0,130.989595,69.681812,-170.000000,-14.000000,28.000000,83.000000,122.000000,178.000000,255.000000,305.000000,389.000000,488,0.0488
current_address_months_count,1000000.0,86.587867,88.406599,-1.000000,0.000000,3.000000,19.000000,52.000000,130.000000,277.000000,370.000000,428.000000,4254,0.4254
session_length_in_minutes,1000000.0,7.544940,8.033106,-1.000000,0.651256,1.253828,3.103053,5.114321,8.866131,21.698864,44.552828,85.899143,2015,0.2015
device_distinct_emails_8w,1000000.0,1.018312,0.180761,-1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000,359,0.0359
velocity_6h,1000000.0,5665.296605,3009.380665,-170.603072,695.593750,1273.448891,3436.365848,5319.769349,7680.717827,11234.629810,13932.187847,16715.565404,0,0.0000


## NUMERIC FEATURE SUMMARY
From this table, intended_balcon_amount has negative values throughout the distribution, so we should not treat negatives there as missing. credit_risk_score and velocity_6h also have legitimate-looking ranges below zero. By contrast, prev_address_months_count, bank_months_count, current_address_months_count, session_length_in_minutes, and device_distinct_emails_8w have -1 sitting outside an otherwise nonnegative range. That makes -1 a strong sentinel-value candidate, but we will not transform it until we verify the dataset definitions.

### 9. Audit categorical and low-cardinality features

In [8]:
low_cardinality_columns = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 12
]

print("Low-cardinality columns:")
print(low_cardinality_columns)

for col in low_cardinality_columns:
    print(f"\n--- {col} ---")
    print(
        df[col]
        .value_counts(dropna=False)
        .sort_index()
    )

Low-cardinality columns:
['fraud_bool', 'income', 'customer_age', 'payment_type', 'employment_status', 'email_is_free', 'housing_status', 'phone_home_valid', 'phone_mobile_valid', 'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'source', 'device_os', 'keep_alive_session', 'device_distinct_emails_8w', 'device_fraud_count', 'month']

--- fraud_bool ---
fraud_bool
0    988971
1     11029
Name: count, dtype: int64

--- income ---
income
0.1    157449
0.2     69345
0.3     50833
0.4     81364
0.5     55858
0.6    111973
0.7    105109
0.8    146650
0.9    221419
Name: count, dtype: int64

--- customer_age ---
customer_age
10     20987
20    245855
30    311433
40    238712
50    140353
60     34770
70      6517
80      1297
90        76
Name: count, dtype: int64

--- payment_type ---
payment_type
AA    258249
AB    370554
AC    252071
AD    118837
AE       289
Name: count, dtype: int64

--- employment_status ---
employment_status
CA    730252
CB    138288
CC     37758
CD     2

## CATEGORICAL & LOW-CARDINALITY FEATURE SUMMARY
A few things stand out. device_fraud_count is confirmed to be completely constant at 0, so it carries no predictive information in Base.csv; we’ll document it as a likely drop candidate rather than removing it yet. month runs from 0 through 7, confirming eight sequential periods are available for a possible chronological split. We also see rare categories such as payment_type=AE, employment_status=CG, housing_status=BG, and source=TELEAPP, which will matter later when we design encoding and validation. income, customer_age, and proposed_credit_limit are numerical in dtype but clearly behave more like discrete/binned variables, which is worth documenting before preprocessing.

### 10. Check fraud rate by month

In [9]:
monthly_fraud_summary = (
    df.groupby("month")["fraud_bool"]
    .agg(
        total_records="count",
        fraud_cases="sum",
        fraud_rate="mean"
    )
)

monthly_fraud_summary["fraud_percent"] = (
    monthly_fraud_summary["fraud_rate"] * 100
)

display(monthly_fraud_summary)

,total_records,fraud_cases,fraud_rate,fraud_percent
month,,,,
0,132440,1500,0.011326,1.132588
1,127620,1198,0.009387,0.938724
2,136979,1198,0.008746,0.874587
3,150936,1392,0.009222,0.922245
4,127691,1452,0.011371,1.137120
5,119323,1411,0.011825,1.182505
6,108168,1450,0.013405,1.340507
7,96843,1428,0.014746,1.474552


## FRAUD RATE SUMMARY
This is a meaningful result. The fraud rate is not stable over time.
It falls from about 1.13% in month 0 to 0.87% in month 2, then rises steadily, reaching about 1.47% in month 7. That is roughly a 69% increase from month 2 to month 7.
This gives us evidence of temporal distribution shift. It strengthens the case for eventually testing a chronological split, because a production fraud model would normally train on earlier activity and encounter later activity. We still will not lock that decision until Milestone 02 examines feature drift and leakage.

### 11. Check for duplicate records

In [10]:
duplicate_count = df.duplicated().sum()
unique_row_count = len(df) - duplicate_count

print(f"Total rows: {len(df):,}")
print(f"Unique rows: {unique_row_count:,}")
print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_count / len(df) * 100:.4f}%")

Total rows: 1,000,000
Unique rows: 1,000,000
Exact duplicate rows: 0
Duplicate percentage: 0.0000%


## DUPLICATE RECORDS SUMMARY
We do not have identical full records repeated in the dataset.

### 12. Check duplicate feature patterns excluding the target

In [11]:
feature_columns = [col for col in df.columns if col != "fraud_bool"]

duplicate_feature_mask = df.duplicated(
    subset=feature_columns,
    keep=False
)

duplicate_feature_rows = df.loc[duplicate_feature_mask]

print(f"Rows sharing identical feature values: {len(duplicate_feature_rows):,}")

if len(duplicate_feature_rows) > 0:
    conflicting_groups = (
        duplicate_feature_rows
        .groupby(feature_columns, dropna=False)["fraud_bool"]
        .nunique()
    )

    conflicting_groups = conflicting_groups[
        conflicting_groups > 1
    ]

    print(f"Feature patterns with conflicting labels: {len(conflicting_groups):,}")
else:
    print("No duplicate feature patterns found.")

Rows sharing identical feature values: 0
No duplicate feature patterns found.


So far, duplicate-based leakage does not appear to be an issue.

## Interpretation Checkpoint 1

Based on the audit so far, answer the following in your own words:

1. Why would accuracy be a misleading primary metric for this dataset?

2. Why do the `-1` values require investigation even though Pandas reported zero missing values?

3. What does the change in fraud rate across `month` suggest about how we might eventually split the data?

4. Why is `device_fraud_count` unlikely to help the model?

5. What did the duplicate checks tell us about possible leakage or conflicting labels?

## Checkpoint 1 Answers

1. **Why would accuracy be a misleading primary metric for this dataset?**

Accuracy would be misleading because the dataset is highly imbalanced. Only about 1.1% of the records are fraud, while about 98.9% are non-fraud. A model could predict “not fraud” for every row and still achieve almost 99% accuracy, even though it would fail to detect any fraud. That is why metrics such as PR-AUC, recall, precision, F1-score, false positives, and false negatives are more useful.

2. **Why do the -1 values require investigation even though Pandas reported zero missing values?**

Pandas only reports standard missing values such as NaN. Some columns appear to use -1 as a special code for unknown, unavailable, or not applicable. For example, prev_address_months_count is -1 for more than 71% of the dataset. Because -1 is still technically a number, Pandas does not count it as missing. We need to verify the meaning of these values before deciding whether to keep, replace, or encode them differently.

3. **What does the change in fraud rate across month suggest about how we might eventually split the data?**

The fraud rate changes over time, from below 0.9% in some earlier months to almost 1.5% in month 7. This suggests that the data distribution may be changing over time. Because of that, a chronological split may better simulate a real production environment, where the model is trained on earlier data and tested on later activity. We still need more analysis before finalizing the split.

4. **Why is device_fraud_count unlikely to help the model?**

device_fraud_count has the same value, 0, for all 1,000,000 rows. Since there is no variation in the feature, it cannot help the model distinguish fraud from non-fraud. It will likely become a drop candidate during preprocessing.

5. **What did the duplicate checks tell us about possible leakage or conflicting labels?**

We found no exact duplicate rows and no rows with identical predictor values. That means we do not currently see evidence of duplicate records appearing multiple times or identical feature patterns receiving conflicting fraud labels. This reduces one potential source of leakage, although we still need to investigate other leakage risks such as future information, feature timing, and temporal relationships.

### Detect constant and near-constant features

In [12]:
feature_variation_summary = []

for col in df.columns:
    value_counts = df[col].value_counts(dropna=False)

    most_common_count = value_counts.iloc[0]
    most_common_percent = most_common_count / len(df) * 100

    feature_variation_summary.append({
        "column": col,
        "unique_count": df[col].nunique(dropna=False),
        "most_common_value": value_counts.index[0],
        "most_common_count": most_common_count,
        "most_common_percent": most_common_percent,
    })

feature_variation_summary = pd.DataFrame(feature_variation_summary)

constant_features = feature_variation_summary[
    feature_variation_summary["unique_count"] == 1
]

near_constant_features = feature_variation_summary[
    (feature_variation_summary["unique_count"] > 1)
    & (feature_variation_summary["most_common_percent"] >= 99)
].sort_values("most_common_percent", ascending=False)

print("Constant features:")
display(constant_features)

print("\nNear-constant features (>= 99% same value):")
display(near_constant_features)

Constant features:


,column,unique_count,most_common_value,most_common_count,most_common_percent
30,device_fraud_count,1,0,1000000,100.0



Near-constant features (>= 99% same value):


,column,unique_count,most_common_value,most_common_count,most_common_percent
25,source,2,INTERNET,992952,99.2952


## CONSTANT CHECK SUMMARY
device_fraud_count is truly constant: every one of the 1,000,000 rows contains 0. We can confidently mark it as a preprocessing drop candidate because it contains no information.
source is near-constant, with INTERNET representing 99.2952% of records. But we should not drop it just because it is rare. The smaller TELEAPP group could still have a very different fraud rate and therefore contain useful predictive information.

In [13]:
source_fraud_summary = (
    df.groupby("source")["fraud_bool"]
    .agg(
        total_records="count",
        fraud_cases="sum",
        fraud_rate="mean"
    )
)

source_fraud_summary["fraud_percent"] = (
    source_fraud_summary["fraud_rate"] * 100
)

display(source_fraud_summary)

,total_records,fraud_cases,fraud_rate,fraud_percent
source,,,,
INTERNET,992952,10917,0.010994,1.099449
TELEAPP,7048,112,0.015891,1.589103


### 
Although TELEAPP represents only 7,048 of the 1,000,000 records, its fraud rate is about 1.59%, compared with about 1.10% for INTERNET. That is roughly a 45% higher relative fraud rate. So source may contain useful predictive signal despite being highly imbalanced.
We will therefore keep source as a candidate feature for now. device_fraud_count, on the other hand, remains a strong drop candidate because it is completely constant.

### Compare fraud rates across categorical features

In [14]:
categorical_columns = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]

for col in categorical_columns:
    category_fraud_summary = (
        df.groupby(col)["fraud_bool"]
        .agg(
            total_records="count",
            fraud_cases="sum",
            fraud_rate="mean"
        )
        .sort_values("fraud_rate", ascending=False)
    )

    category_fraud_summary["fraud_percent"] = (
        category_fraud_summary["fraud_rate"] * 100
    )

    print(f"\n--- Fraud rate by {col} ---")
    display(category_fraud_summary)


--- Fraud rate by payment_type ---


,total_records,fraud_cases,fraud_rate,fraud_percent
payment_type,,,,
AC,252071,4209,0.016698,1.669768
AB,370554,4169,0.011251,1.125072
AD,118837,1286,0.010822,1.082155
AA,258249,1364,0.005282,0.528172
AE,289,1,0.003460,0.346021



--- Fraud rate by employment_status ---


,total_records,fraud_cases,fraud_rate,fraud_percent
employment_status,,,,
CC,37758,932,0.024684,2.468351
CG,453,7,0.015453,1.545254
CA,730252,8899,0.012186,1.218620
CB,138288,953,0.006891,0.689142
CD,26522,100,0.003770,0.377045
CE,22693,53,0.002336,0.233552
CF,44034,85,0.001930,0.193033



--- Fraud rate by housing_status ---


,total_records,fraud_cases,fraud_rate,fraud_percent
housing_status,,,,
BA,169675,6357,0.037466,3.746574
BD,26161,226,0.008639,0.863881
BC,372143,2288,0.006148,0.614817
BB,260965,1568,0.006008,0.600847
BF,1669,7,0.004194,0.419413
BG,252,1,0.003968,0.396825
BE,169135,582,0.003441,0.344104



--- Fraud rate by source ---


,total_records,fraud_cases,fraud_rate,fraud_percent
source,,,,
TELEAPP,7048,112,0.015891,1.589103
INTERNET,992952,10917,0.010994,1.099449



--- Fraud rate by device_os ---


,total_records,fraud_cases,fraud_rate,fraud_percent
device_os,,,,
windows,263506,6507,0.024694,2.469393
macintosh,53826,752,0.013971,1.397094
x11,7228,81,0.011206,1.120642
other,342728,1974,0.005760,0.575967
linux,332712,1715,0.005155,0.515461


## CATEGORICAL FEATURE SUMMARY 
These categorical features show substantial differences in fraud prevalence, which tells us they may carry useful signal—but we should interpret them cautiously because some categories are very small.
A few examples stand out: housing_status=BA has about 3.75% fraud, far above the overall 1.10%; employment_status=CC and device_os=windows are both around 2.47%. At the other end, several categories are below 0.5%. Rare groups such as payment_type=AE, employment_status=CG, and housing_status=BG need extra caution because their rates are based on very small samples. These differences are associations, not proof that the categories cause fraud.

### Audit binary indicator features

In [15]:
binary_columns = [
    "email_is_free",
    "phone_home_valid",
    "phone_mobile_valid",
    "has_other_cards",
    "foreign_request",
    "keep_alive_session",
]

for col in binary_columns:
    binary_fraud_summary = (
        df.groupby(col)["fraud_bool"]
        .agg(
            total_records="count",
            fraud_cases="sum",
            fraud_rate="mean"
        )
    )

    binary_fraud_summary["fraud_percent"] = (
        binary_fraud_summary["fraud_rate"] * 100
    )

    print(f"\n--- Fraud rate by {col} ---")
    display(binary_fraud_summary)


--- Fraud rate by email_is_free ---


,total_records,fraud_cases,fraud_rate,fraud_percent
email_is_free,,,,
0,470114,3738,0.007951,0.795126
1,529886,7291,0.013760,1.375956



--- Fraud rate by phone_home_valid ---


,total_records,fraud_cases,fraud_rate,fraud_percent
phone_home_valid,,,,
0,582923,8238,0.014132,1.413223
1,417077,2791,0.006692,0.669181



--- Fraud rate by phone_mobile_valid ---


...

,total_records,fraud_cases,fraud_rate,fraud_percent
has_other_cards,,,,
0,777012,10098,0.012996,1.299594
1,222988,931,0.004175,0.417511



--- Fraud rate by foreign_request ---


,total_records,fraud_cases,fraud_rate,fraud_percent
foreign_request,,,,
0,974758,10474,0.010745,1.074523
1,25242,555,0.021987,2.198716



--- Fraud rate by keep_alive_session ---


,total_records,fraud_cases,fraud_rate,fraud_percent
keep_alive_session,,,,
0,423053,7261,0.017163,1.716333
1,576947,3768,0.006531,0.653093


## BINARY INDICATOR FEATURE SUMMARY
The largest differences are especially notable. has_other_cards=0 has about 1.30% fraud versus 0.42% when the value is 1, roughly a 3.1× difference. keep_alive_session=0 is about 1.72% versus 0.65%, and foreign_request=1 is about 2.20% versus 1.07%. Invalid home and mobile phone indicators also correspond to higher observed fraud rates, while email_is_free=1 has a higher rate than 0.

### Compare numerical features for fraud vs. non-fraud

In [16]:
numeric_feature_columns = [
    col for col in df.select_dtypes(include=np.number).columns
    if col not in ["fraud_bool", "month"]
]

numeric_target_summary = (
    df.groupby("fraud_bool")[numeric_feature_columns]
    .median()
    .T
)

numeric_target_summary.columns = [
    "median_non_fraud",
    "median_fraud"
]

numeric_target_summary["absolute_difference"] = (
    numeric_target_summary["median_fraud"]
    - numeric_target_summary["median_non_fraud"]
).abs()

numeric_target_summary = numeric_target_summary.sort_values(
    "absolute_difference",
    ascending=False
)

display(numeric_target_summary)

,median_non_fraud,median_fraud,absolute_difference
velocity_6h,5324.540439,4916.464728,408.075711
proposed_credit_limit,200.000000,500.000000,300.000000
velocity_4w,4914.578888,4827.695220,86.883668
velocity_24h,4751.027067,4665.725851,85.301217
zip_count_4w,1262.000000,1328.000000,66.000000
credit_risk_score,122.000000,178.000000,56.000000
current_address_months_count,52.000000,94.000000,42.000000
customer_age,30.000000,40.000000,10.000000
bank_branch_count_8w,9.000000,2.000000,7.000000
bank_months_count,5.000000,2.000000,3.000000


## SUMMARY
Several numerical features have noticeably different medians between fraud and non-fraud records. For example, fraud records have a higher median proposed_credit_limit (500 vs. 200), higher credit_risk_score (178 vs. 122), higher current_address_months_count (94 vs. 52), and lower name_email_similarity (~0.29 vs. ~0.49). bank_branch_count_8w also differs substantially, with medians of 2 for fraud versus 9 for non-fraud.
One important caution: we cannot rank feature importance from the absolute_difference column. velocity_6h appears first partly because it is measured on a scale of thousands, while name_email_similarity is constrained to a much smaller range. A difference of 408 in one variable is not automatically more important than a difference of 0.20 in another.
Also notice why median comparisons do not work well for binary variables: both groups may have the same median even though their fraud rates differ substantially. We already saw that with foreign_request, has_other_cards, and the phone indicators.

### Normalize the numeric differences

In [17]:
continuous_numeric_columns = [
    col for col in numeric_feature_columns
    if df[col].nunique(dropna=False) > 12
    and col != "device_fraud_count"
]

normalized_median_summary = []

for col in continuous_numeric_columns:
    non_fraud = df.loc[df["fraud_bool"] == 0, col]
    fraud = df.loc[df["fraud_bool"] == 1, col]

    median_non_fraud = non_fraud.median()
    median_fraud = fraud.median()

    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    normalized_shift = (
        (median_fraud - median_non_fraud) / iqr
        if iqr != 0
        else np.nan
    )

    normalized_median_summary.append({
        "column": col,
        "median_non_fraud": median_non_fraud,
        "median_fraud": median_fraud,
        "iqr": iqr,
        "normalized_median_shift": normalized_shift,
        "absolute_normalized_shift": abs(normalized_shift),
    })

normalized_median_summary = (
    pd.DataFrame(normalized_median_summary)
    .sort_values("absolute_normalized_shift", ascending=False)
)

display(normalized_median_summary)

,column,median_non_fraud,median_fraud,iqr,normalized_median_shift,absolute_normalized_shift
11,credit_risk_score,122.000000,178.000000,95.000000,0.589474,0.589474
0,name_email_similarity,0.493571,0.292420,0.530351,-0.379280,0.379280
2,current_address_months_count,52.000000,94.000000,111.000000,0.378378,0.378378
9,bank_branch_count_8w,9.000000,2.000000,24.000000,-0.291667,0.291667
10,date_of_birth_distinct_emails_4w,9.000000,7.000000,7.000000,-0.285714,0.285714
12,bank_months_count,5.000000,2.000000,26.000000,-0.115385,0.115385
3,days_since_request,0.015203,0.013094,0.019137,-0.110234,0.110234
6,velocity_6h,5324.540439,4916.464728,4244.351979,-0.096146,0.096146
8,velocity_4w,4914.578888,4827.695220,1219.714934,-0.071233,0.071233
5,zip_count_4w,1262.000000,1328.000000,1050.000000,0.062857,0.062857


## NORMALIZATION SUMMARY
This normalized view is much more informative. credit_risk_score shows the largest median shift relative to its own spread, followed by name_email_similarity, current_address_months_count, bank_branch_count_8w, and date_of_birth_distinct_emails_4w. Those are strong candidates for deeper investigation later, but this still is not feature importance.
One especially important lesson is prev_address_months_count. Its normalized shift is 0 because the median for both fraud and non-fraud is -1. Since over 71% of the column is -1, the sentinel value dominates the median. That does not mean the feature has no predictive value. We will eventually analyze the sentinel state separately from the valid numeric values.

## Numeric Feature Audit Interpretation

After normalizing median differences by each feature's interquartile range, several numerical features show noticeable differences between fraud and non-fraud records.

The largest normalized median differences were observed for:

- `credit_risk_score`
- `name_email_similarity`
- `current_address_months_count`
- `bank_branch_count_8w`
- `date_of_birth_distinct_emails_4w`

These results suggest that the distributions of these features differ between fraud and non-fraud records and should be examined more closely during later modeling and explainability analysis.

The normalized differences should not be interpreted as feature importance or evidence of causation. They are exploratory comparisons only.

`prev_address_months_count` produced a normalized median difference of zero because both fraud and non-fraud groups have a median value of `-1`. Since `-1` occurs in more than 71% of this feature, the sentinel value dominates the median. This means the feature requires separate analysis of the `-1` state and the valid non-negative values before determining its usefulness.

Several binary features also showed zero median differences even though their fraud rates differed meaningfully. This demonstrates why different feature types require different exploratory methods.

### Test whether -1 sentinel states are associated with fraud

In [18]:
sentinel_columns = [
    "prev_address_months_count",
    "bank_months_count",
    "current_address_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w",
]

sentinel_fraud_results = []

for col in sentinel_columns:
    sentinel_mask = df[col] == -1

    sentinel_count = sentinel_mask.sum()
    normal_count = (~sentinel_mask).sum()

    sentinel_fraud_rate = df.loc[sentinel_mask, "fraud_bool"].mean()
    normal_fraud_rate = df.loc[~sentinel_mask, "fraud_bool"].mean()

    sentinel_fraud_results.append({
        "column": col,
        "sentinel_count": sentinel_count,
        "sentinel_percent": sentinel_count / len(df) * 100,
        "fraud_rate_when_minus_one": sentinel_fraud_rate,
        "fraud_percent_when_minus_one": sentinel_fraud_rate * 100,
        "fraud_rate_when_not_minus_one": normal_fraud_rate,
        "fraud_percent_when_not_minus_one": normal_fraud_rate * 100,
    })

sentinel_fraud_summary = pd.DataFrame(sentinel_fraud_results)

display(sentinel_fraud_summary)

,column,sentinel_count,sentinel_percent,fraud_rate_when_minus_one,fraud_percent_when_minus_one,fraud_rate_when_not_minus_one,fraud_percent_when_not_minus_one
0,prev_address_months_count,712920,71.2920,0.014215,1.421478,0.003118,0.311760
1,bank_months_count,253635,25.3635,0.016323,1.632267,0.009230,0.923007
2,current_address_months_count,4254,0.4254,0.003291,0.329102,0.011062,1.106206
3,session_length_in_minutes,2015,0.2015,0.008933,0.893300,0.011033,1.103323
4,device_distinct_emails_8w,359,0.0359,0.011142,1.114206,0.011029,1.102896


## Sentinel Value Interpretation

Several numerical features contain `-1` values that appear to represent special states rather than ordinary measurements.

The relationship between the `-1` state and fraud differs substantially by feature:

- `prev_address_months_count = -1` has a much higher fraud rate than valid values.
- `bank_months_count = -1` also has a higher fraud rate than valid values.
- `current_address_months_count = -1` has a lower fraud rate than valid values.
- `session_length_in_minutes = -1` shows a modestly lower fraud rate.
- `device_distinct_emails_8w = -1` has nearly the same fraud rate as valid values.

This means the `-1` state should not automatically be replaced with a median or treated identically across all features.

During preprocessing, each sentinel-coded feature should be handled based on its documented meaning and observed behavior. Possible strategies may include preserving the sentinel, creating an explicit missing/unknown indicator, or transforming the feature after the train/validation/test split is finalized.

These associations do not establish causation and must not be interpreted as evidence that missing or unknown information causes fraud.

### Build the feature review table

In [19]:
sentinel_candidates = {
    "prev_address_months_count",
    "bank_months_count",
    "current_address_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w",
}

binary_features = {
    "email_is_free",
    "phone_home_valid",
    "phone_mobile_valid",
    "has_other_cards",
    "foreign_request",
    "keep_alive_session",
}

categorical_features = {
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
}

feature_review_rows = []

for col in df.columns:
    unique_count = df[col].nunique(dropna=False)

    if col == "fraud_bool":
        feature_type = "target"
    elif col == "month":
        feature_type = "temporal"
    elif unique_count == 1:
        feature_type = "constant"
    elif col in categorical_features:
        feature_type = "categorical"
    elif col in binary_features:
        feature_type = "binary_indicator"
    elif pd.api.types.is_numeric_dtype(df[col]) and unique_count <= 12:
        feature_type = "discrete_numeric"
    else:
        feature_type = "continuous_numeric"

    audit_flags = []

    if unique_count == 1:
        audit_flags.append("constant")

    if col == "source":
        audit_flags.append("near_constant")

    if col in sentinel_candidates:
        audit_flags.append("possible_-1_sentinel")

    if col == "month":
        audit_flags.append("temporal_split_review")

    if col in categorical_features:
        min_category_count = df[col].value_counts().min()
        if min_category_count < 1000:
            audit_flags.append("rare_category_present")

    feature_review_rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "feature_type": feature_type,
        "unique_count": unique_count,
        "audit_flags": "; ".join(audit_flags) if audit_flags else "none",
    })

feature_review = pd.DataFrame(feature_review_rows)

display(feature_review)

,column,dtype,feature_type,unique_count,audit_flags
0,fraud_bool,int64,target,2,none
1,income,float64,discrete_numeric,9,none
2,name_email_similarity,float64,continuous_numeric,998861,none
3,prev_address_months_count,int64,continuous_numeric,374,possible_-1_sentinel
4,current_address_months_count,int64,continuous_numeric,423,possible_-1_sentinel
5,customer_age,int64,discrete_numeric,9,none
6,days_since_request,float64,continuous_numeric,989330,none
7,intended_balcon_amount,float64,continuous_numeric,994971,none
8,payment_type,str,categorical,5,rare_category_present
9,zip_count_4w,int64,continuous_numeric,6306,none


### Add a recommended next action to the feature review

In [20]:
def assign_next_action(row):
    if row["feature_type"] == "target":
        return "target_only"

    if row["feature_type"] == "constant":
        return "drop_candidate"

    if row["feature_type"] == "temporal":
        return "audit_for_split_strategy"

    if "possible_-1_sentinel" in row["audit_flags"]:
        return "investigate_sentinel_before_preprocessing"

    if "rare_category_present" in row["audit_flags"]:
        return "retain_and_review_rare_categories"

    if "near_constant" in row["audit_flags"]:
        return "retain_and_validate_predictive_value"

    return "retain_candidate"

feature_review["next_action"] = feature_review.apply(
    assign_next_action,
    axis=1
)

display(feature_review)

,column,dtype,feature_type,unique_count,audit_flags,next_action
0,fraud_bool,int64,target,2,none,target_only
1,income,float64,discrete_numeric,9,none,retain_candidate
2,name_email_similarity,float64,continuous_numeric,998861,none,retain_candidate
3,prev_address_months_count,int64,continuous_numeric,374,possible_-1_sentinel,investigate_sentinel_before_preprocessing
4,current_address_months_count,int64,continuous_numeric,423,possible_-1_sentinel,investigate_sentinel_before_preprocessing
5,customer_age,int64,discrete_numeric,9,none,retain_candidate
6,days_since_request,float64,continuous_numeric,989330,none,retain_candidate
7,intended_balcon_amount,float64,continuous_numeric,994971,none,retain_candidate
8,payment_type,str,categorical,5,rare_category_present,retain_and_review_rare_categories
9,zip_count_4w,int64,continuous_numeric,6306,none,retain_candidate


### Export the Milestone 01 feature review

In [21]:
FEATURE_REVIEW_PATH = (
    PROJECT_ROOT
    / "reports"
    / "milestone_01_feature_review.csv"
)

feature_review.to_csv(
    FEATURE_REVIEW_PATH,
    index=False
)

print("Saved feature review to:")
print(FEATURE_REVIEW_PATH)

print("\nRows exported:", len(feature_review))
print("Columns exported:", len(feature_review.columns))
print("File exists:", FEATURE_REVIEW_PATH.exists())

Saved feature review to:
c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\reports\milestone_01_feature_review.csv

Rows exported: 32
Columns exported: 6
File exists: True


### Export a compact Milestone 01 audit summary

In [22]:
audit_metrics = pd.DataFrame([
    {"metric": "row_count", "value": len(df)},
    {"metric": "column_count", "value": df.shape[1]},
    {"metric": "fraud_count", "value": int(df["fraud_bool"].sum())},
    {"metric": "fraud_rate", "value": df["fraud_bool"].mean()},
    {"metric": "standard_missing_values", "value": int(df.isna().sum().sum())},
    {"metric": "exact_duplicate_rows", "value": int(df.duplicated().sum())},
    {
        "metric": "duplicate_feature_pattern_rows",
        "value": int(duplicate_feature_mask.sum())
    },
    {
        "metric": "constant_feature_count",
        "value": int((feature_review["feature_type"] == "constant").sum())
    },
    {
        "metric": "categorical_feature_count",
        "value": int((feature_review["feature_type"] == "categorical").sum())
    },
    {
        "metric": "binary_indicator_count",
        "value": int((feature_review["feature_type"] == "binary_indicator").sum())
    },
    {
        "metric": "sentinel_candidate_count",
        "value": len(sentinel_candidates)
    },
    {
        "metric": "month_count",
        "value": df["month"].nunique()
    },
])

AUDIT_METRICS_PATH = (
    PROJECT_ROOT
    / "reports"
    / "milestone_01_data_audit_metrics.csv"
)

audit_metrics.to_csv(
    AUDIT_METRICS_PATH,
    index=False
)

display(audit_metrics)

print("\nSaved audit metrics to:")
print(AUDIT_METRICS_PATH)
print("File exists:", AUDIT_METRICS_PATH.exists())

,metric,value
0,row_count,1000000.000000
1,column_count,32.000000
2,fraud_count,11029.000000
3,fraud_rate,0.011029
4,standard_missing_values,0.000000
5,exact_duplicate_rows,0.000000
6,duplicate_feature_pattern_rows,0.000000
7,constant_feature_count,1.000000
8,categorical_feature_count,5.000000
9,binary_indicator_count,6.000000



Saved audit metrics to:
c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\reports\milestone_01_data_audit_metrics.csv
File exists: True


## Milestone 01 Data Audit Conclusion

The initial audit of `Base.csv` confirms that the dataset contains 1,000,000 rows and 32 columns, with `fraud_bool` as the binary target.

Fraud is highly imbalanced, representing approximately 1.10% of all records. Because of this imbalance, accuracy will not be used as the primary model-selection metric. PR-AUC will be prioritized, with precision, recall, F1-score, false positives, false negatives, ROC-AUC, calibration, and threshold-dependent business cost used as supporting metrics.

No standard Pandas missing values, exact duplicate rows, or duplicate predictor patterns were found. However, five features contain possible `-1` sentinel values that require further investigation before preprocessing.

The dataset contains one constant feature, `device_fraud_count`, which is a likely drop candidate. The `source` feature is near-constant but shows different fraud rates between its categories, so it will remain under consideration.

Several categorical and binary features show substantial differences in fraud prevalence across groups. These relationships are exploratory associations and must not be interpreted as causal effects.

The fraud rate also changes across the eight values of `month`, indicating possible temporal distribution shift. A chronological split may therefore better represent future deployment than a random split, but the split strategy will not be finalized until Milestone 02 completes the leakage and temporal-distribution review.

No preprocessing, feature dropping, scaling, encoding, resampling, threshold selection, or model training has been performed in this notebook.

### Deferred Decisions for Milestone 02 and Milestone 03

- Verify the documented meaning of sentinel-coded values.
- Perform a formal feature leakage review.
- Evaluate feature availability at prediction time.
- Analyze feature distribution changes across `month`.
- Compare chronological and stratified split strategies.
- Define and seal train, validation, and final test partitions.
- Finalize feature inclusion/exclusion decisions.
- Design leakage-safe categorical and numerical preprocessing.
- Preserve the final test set for one-time evaluation after model-development decisions are frozen.